## Section 1 - Setup

In [ ]:
!pip install -q trl transformers datasets peft requests matplotlib tabulate

## Section 2 - Connect to Environment
Test all 5 environments with curl-equivalent requests
Print PASS/FAIL for each

In [ ]:
import requests

tasks = ['easy', 'medium', 'hard', 'adversarial', 'execution']
print("Testing environment connections...")

for task in tasks:
    try:
        # The reset endpoint initiates the task
        res = requests.post(f"http://localhost:7860/reset?task_id={task}", timeout=5)
        if res.status_code == 200:
            print(f"Task '{task}': PASS")
        else:
            print(f"Task '{task}': FAIL (Status {res.status_code})")
    except Exception as e:
        print(f"Task '{task}': FAIL ({e})")


## Section 3 - Baseline Evaluation
Run untrained Qwen2.5-0.5B on easy/medium/hard
Record scores in a dict: baseline_scores

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import requests

print("Loading model for baseline evaluation...")
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

def evaluate_model(task_id, num_samples=3):
    # Dummy prompt for baseline to check scoring
    prompt_text = "You are a legal contract analyst. Find all contradictions. Respond with JSON."
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    total_score = 0.0
    for _ in range(num_samples):
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128, pad_token_id=tokenizer.pad_token_id)
        completion = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        try:
            requests.post(f"http://localhost:7860/reset?task_id={task_id}", timeout=2)
            resp = requests.post("http://localhost:7860/step", json={"action": completion}, timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                score = float(data.get("reward", data.get("score", 0.0)))
            else:
                score = 0.0
        except:
            score = 0.0
        total_score += score
        
    return total_score / num_samples

baseline_scores = {}
tasks_to_eval = ["easy", "medium", "hard"]

print("Running baseline evaluation...")
for task in tasks_to_eval:
    score = evaluate_model(task)
    baseline_scores[task] = score
    print(f"Baseline for {task}: {score:.4f}")


## Section 4 - GRPO Training
Run train_grpo.py logic inline
Show live reward curve updating

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
from transformers import TrainerCallback

class LivePlotCallback(TrainerCallback):
    def __init__(self):
        self.steps = []
        self.rewards = []
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None: return
        reward_keys = [k for k in logs.keys() if 'reward' in k.lower() and isinstance(logs[k], (int, float))]
        if reward_keys:
            self.rewards.append(logs[reward_keys[0]])
            self.steps.append(state.global_step)
            
            clear_output(wait=True)
            plt.figure(figsize=(10, 5))
            plt.plot(self.steps, self.rewards, marker='o', label='Reward')
            plt.title('Live Training Reward')
            plt.xlabel('Steps')
            plt.ylabel('Reward')
            plt.grid(True)
            plt.legend()
            plt.show()

def clausr_reward(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion[0]['content'] if isinstance(completion, list) else completion
        try:
            requests.post("http://localhost:7860/reset?task_id=easy", timeout=2)
            resp = requests.post("http://localhost:7860/step", json={"action": text}, timeout=5)
            score = float(resp.json().get("reward", resp.json().get("score", 0.0))) if resp.status_code == 200 else 0.0
        except:
            score = 0.0
        rewards.append(score)
    return rewards

sample_clauses = [
    "1: The vendor provides services on Monday. 2: The vendor shall not work on Mondays.",
    "1: Payment is due in 30 days. 2: Payment must be completed immediately.",
    "1: Confidentiality lasts for 2 years. 2: Confidentiality is perpetual.",
    "1: Either party can terminate with 30 days notice. 2: Contract is non-terminable.",
    "1: Disputes settled in NY. 2: All litigation must occur in CA."
]

data = []
for clauses in sample_clauses * 10:
    prompt_text = f"You are a legal contract analyst. Here are the clauses: {clauses}. Find all contradictions. Respond with JSON: {{'findings': []}}"
    data.append({"prompt": [{"role": "user", "content": prompt_text}]})

dataset = Dataset.from_list(data)

training_args = GRPOConfig(
    output_dir="clausr-grpo-colab",
    num_train_epochs=1,
    max_completion_length=128,
    num_generations=2,
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[clausr_reward],
    args=training_args,
    train_dataset=dataset,
    callbacks=[LivePlotCallback()]
)

print("Starting GRPO Training...")
trainer.train()


## Section 5 - Post-Training Evaluation
Run same model on easy/medium/hard
Print improvement table

In [ ]:
import pandas as pd

after_scores = {}
print("Running post-training evaluation...")
for task in tasks_to_eval:
    score = evaluate_model(task)
    after_scores[task] = score

results = []
for task in tasks_to_eval:
    before = baseline_scores[task]
    after = after_scores[task]
    delta = after - before
    results.append({
        "Task": task.capitalize(), 
        "Before": f"{before:.2f}", 
        "After": f"{after:.2f}", 
        "Delta": f"{delta:+.2f}"
    })

df = pd.DataFrame(results)
print("\nImprovement Table:")
print(df.to_markdown(index=False))


## Section 6 - Save all 4 plots as PNG files

In [ ]:
import matplotlib.pyplot as plt

# 1. Training Reward Plot
history = trainer.state.log_history
steps = []
rewards = []
for log in history:
    reward_keys = [k for k in log.keys() if 'reward' in k.lower() and isinstance(log[k], (int, float))]
    if reward_keys:
        rewards.append(log[reward_keys[0]])
        steps.append(log.get('step', len(steps)))

fig1 = plt.figure(figsize=(8,5))
if rewards:
    plt.plot(steps, rewards, marker='o')
plt.title("Plot 1: Training Reward Curve")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.grid(True)
plt.savefig("plot_1_training_reward.png")
plt.close(fig1)

def plot_comparison(task_name, before, after, plot_name, title):
    fig = plt.figure(figsize=(6,4))
    plt.bar(['Before', 'After'], [before, after], color=['red', 'green'])
    plt.title(title)
    plt.ylabel("Score")
    plt.ylim(0, 1.0)
    plt.savefig(plot_name)
    plt.close(fig)

# 2. Easy Task Plot
plot_comparison('Easy', baseline_scores['easy'], after_scores['easy'], "plot_2_easy_task.png", "Plot 2: Easy Task Performance")

# 3. Medium Task Plot
plot_comparison('Medium', baseline_scores['medium'], after_scores['medium'], "plot_3_medium_task.png", "Plot 3: Medium Task Performance")

# 4. Hard Task Plot
plot_comparison('Hard', baseline_scores['hard'], after_scores['hard'], "plot_4_hard_task.png", "Plot 4: Hard Task Performance")

print("Saved all 4 plots:")
print("- plot_1_training_reward.png")
print("- plot_2_easy_task.png")
print("- plot_3_medium_task.png")
print("- plot_4_hard_task.png")
